# SigFlow-Sim: Signature-Conditioned Normalizing Flow for Market Return Simulation

This notebook implements the full SigFlow-Sim pipeline:
- **Phase 1:** Path Signature extraction (Rough Path Theory)
- **Phase 2:** Conditional Normalizing Flow (Generation Engine)
- **Phase 3:** Custom Loss Judges (MLE + Distribution + ACF + Diversity)
- **Phase 4:** Training Loop + Scenario Simulation

> ⚠️ This is a research/educational tool. Not financial advice.

## 0. Install Dependencies

In [ ]:
# Install required packages
!pip install iisignatures nflows yfinance torch numpy pandas matplotlib scipy --quiet

## 1. Imports & Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import iisignatures
import yfinance as yf

from nflows.flows import Flow
from nflows.distributions import StandardNormal
from nflows.transforms import (
    CompositeTransform,
    MaskedAffineAutoregressiveTransform,
    RandomPermutation,
)

from scipy.stats import kstest, norm

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Device ───────────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

# ── Hyperparameters ───────────────────────────────────────────────────────────
CONFIG = {
    # Data
    'ticker'        : 'SPY',      # Stock to model
    'start_date'    : '2010-01-01',
    'end_date'      : '2024-01-01',

    # Phase 1 – Path Signature
    'window_size'   : 20,         # Rolling window (days)
    'sig_depth'     : 3,          # Truncation depth M

    # Phase 2 – Normalizing Flow
    'n_flow_layers' : 6,          # Number of invertible transforms
    'hidden_dim'    : 128,        # Hidden units per layer
    'context_dim'   : None,       # Set automatically from sig size

    # Phase 3 – Loss weights (λ)
    'lambda_dist'   : 1.0,        # λ₁: Distribution judge
    'lambda_acf'    : 0.5,        # λ₂: Autocorrelation judge
    'lambda_div'    : 0.1,        # λ₃: Diversity / anti-collapse judge

    # Phase 4 – Training
    'epochs'        : 300,
    'batch_size'    : 256,
    'learning_rate' : 1e-3,
    'log_every'     : 50,

    # Simulation
    'n_scenarios'   : 50,         # Number of future paths to generate
    'n_steps'       : 30,         # Days to simulate forward
}

print("Configuration loaded ✓")
print(pd.DataFrame.from_dict(CONFIG, orient='index', columns=['Value']).to_string())

---
## Phase 1 — The Data & The "Memory" (Rough Path / Signature Theory)

We compute the truncated **path signature** of rolling windows of log-returns.

For a path $X:[0,T]\to\mathbb{R}^d$ augmented with time $\hat{X}_t = (t, X_t)$, the signature is:

$$S(X)_{a,b} = \left(1,\;\int_a^b d\hat{X}_{t_1},\;\int_a^b\int_a^{t_2}d\hat{X}_{t_1}\otimes d\hat{X}_{t_2},\;\dots\right)$$

Truncated at depth $M=3$ this gives a fixed-length feature vector.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 1-A  Download price data
# ─────────────────────────────────────────────────────────────────
print(f"Downloading {CONFIG['ticker']} price data...")
raw = yf.download(CONFIG['ticker'],
                  start=CONFIG['start_date'],
                  end=CONFIG['end_date'],
                  progress=False)

prices = raw['Close'].dropna()
print(f"  Downloaded {len(prices)} trading days  ({prices.index[0].date()} → {prices.index[-1].date()})")

# Log-returns:  r_t = ln(P_t / P_{t-1})
log_returns = np.log(prices / prices.shift(1)).dropna().values.astype(np.float32)
print(f"  Log-return series length: {len(log_returns)}")

# Quick sanity plot
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(prices.values, lw=0.8, color='steelblue')
axes[0].set_title(f'{CONFIG["ticker"]} Close Price')
axes[0].set_xlabel('Trading Days')
axes[0].set_ylabel('Price (USD)')

axes[1].plot(log_returns, lw=0.5, color='darkorange', alpha=0.8)
axes[1].set_title('Log-Returns')
axes[1].set_xlabel('Trading Days')
axes[1].set_ylabel('Log-Return')
plt.tight_layout()
plt.savefig('phase1_data.png', dpi=120, bbox_inches='tight')
plt.show()
print("\nPhase 1-A complete ✓")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 1-B  Build Path Signatures
# ─────────────────────────────────────────────────────────────────
def build_path_with_time(window: np.ndarray) -> np.ndarray:
    """
    Given a 1-D array of returns of length W, return an augmented path
    of shape (W, 2):  [ [t_0, X_0], [t_1, X_1], ..., [t_{W-1}, X_{W-1}] ]

    Adding the time dimension makes the signature invariant to time-warping
    (the AI cares about the sequence, not the absolute clock).
    """
    T = len(window)
    time_vec = np.linspace(0, 1, T, dtype=np.float32)   # normalised [0,1]
    path = np.stack([time_vec, window], axis=1)          # shape (T, 2)
    return path


def compute_signature(path: np.ndarray, depth: int) -> np.ndarray:
    """
    Compute the truncated iterated-integral signature of `path` up to
    depth `depth` using the iisignatures library.

    Returns a 1-D numpy array (the signature vector).
    """
    # iisignatures expects shape (1, T, d) for a single path
    path_3d = path[np.newaxis, :, :]          # (1, T, 2)
    sig = iisignatures.sig(path_3d, depth)    # (1, sig_size)
    return sig[0]                             # (sig_size,)


# Compute signature size for our (depth=3, d=2) setting
d = 2  # path dimension (time + return)
M = CONFIG['sig_depth']
sig_size = iisignatures.siglength(d, M)
CONFIG['context_dim'] = sig_size
print(f"Signature dimension for d={d}, depth={M}: {sig_size}")

# ── Rolling window loop ───────────────────────────────────────────
W = CONFIG['window_size']
N = len(log_returns)

sig_list    = []
target_list = []

for i in range(W, N):
    window  = log_returns[i - W : i]       # last W returns (the "past")
    target  = log_returns[i]               # next return  (the "future")

    path    = build_path_with_time(window)
    sig_vec = compute_signature(path, M)

    sig_list.append(sig_vec)
    target_list.append(target)

signatures = np.array(sig_list,    dtype=np.float32)  # (N-W, sig_size)
targets    = np.array(target_list, dtype=np.float32)  # (N-W,)

print(f"\nSignature matrix shape : {signatures.shape}")
print(f"Target vector shape    : {targets.shape}")
print(f"\nSample signature vector (first 10 elements):\n  {signatures[0, :10]}")
print("\nPhase 1-B complete ✓")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 1-C  Normalise signatures & build PyTorch datasets
# ─────────────────────────────────────────────────────────────────
from sklearn.preprocessing import StandardScaler

# Train / validation split (80 / 20)
split = int(0.8 * len(signatures))

sig_train, sig_val   = signatures[:split], signatures[split:]
tgt_train, tgt_val   = targets[:split],    targets[split:]

# Fit scaler only on training data (avoid look-ahead bias)
scaler = StandardScaler()
sig_train_scaled = scaler.fit_transform(sig_train).astype(np.float32)
sig_val_scaled   = scaler.transform(sig_val).astype(np.float32)

# Convert to tensors
def to_tensor(arr): return torch.tensor(arr, dtype=torch.float32).to(DEVICE)

sig_train_t = to_tensor(sig_train_scaled)
tgt_train_t = to_tensor(tgt_train.reshape(-1, 1))
sig_val_t   = to_tensor(sig_val_scaled)
tgt_val_t   = to_tensor(tgt_val.reshape(-1, 1))

train_dataset = TensorDataset(sig_train_t, tgt_train_t)
val_dataset   = TensorDataset(sig_val_t,   tgt_val_t)

train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=CONFIG['batch_size'], shuffle=False)

print(f"Train samples : {len(train_dataset)}")
print(f"Val   samples : {len(val_dataset)}")
print("Phase 1-C complete ✓")

---
## Phase 2 — The Generation Engine (Conditional Normalizing Flow)

We map Gaussian noise $Z \sim \mathcal{N}(0,I)$ through an invertible, differentiable network conditioned on the Signature $S(X)$:

$$Y_{\text{fake}} = f_\theta(Z \mid S(X))$$

The training log-likelihood uses the **Change of Variables** formula:

$$\log p_Y(y \mid S(X)) = \log p_Z(f_\theta^{-1}(y \mid S(X))) + \log\left|\det\left(\frac{\partial f_\theta^{-1}}{\partial y}\right)\right|$$

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 2-A  Build the Conditional Normalizing Flow
# ─────────────────────────────────────────────────────────────────
def build_conditional_flow(
    context_dim: int,
    n_layers   : int,
    hidden_dim : int,
    data_dim   : int = 1,
) -> Flow:
    """
    Build a Conditional Normalizing Flow with `n_layers` Masked Autoregressive
    transforms interleaved with random permutations.

    The context (= Signature vector) is passed into each transform so the
    network can condition its warping on the preceding 20-day history.

    Architecture:
      Base distribution : StandardNormal(data_dim=1)
      Transforms        : [Permute → MAF] × n_layers
    """
    transforms = []
    for _ in range(n_layers):
        transforms.append(RandomPermutation(features=data_dim))
        transforms.append(
            MaskedAffineAutoregressiveTransform(
                features          = data_dim,
                hidden_features   = hidden_dim,
                context_features  = context_dim,   # <── conditioned on Sig
                num_blocks        = 2,
                use_residual_blocks=True,
            )
        )

    transform    = CompositeTransform(transforms)
    distribution = StandardNormal(shape=[data_dim])
    flow         = Flow(transform=transform, distribution=distribution)
    return flow


flow = build_conditional_flow(
    context_dim = CONFIG['context_dim'],
    n_layers    = CONFIG['n_flow_layers'],
    hidden_dim  = CONFIG['hidden_dim'],
).to(DEVICE)

n_params = sum(p.numel() for p in flow.parameters() if p.requires_grad)
print(f"Flow architecture built ✓")
print(f"  Trainable parameters : {n_params:,}")
print(f"  Context dimension    : {CONFIG['context_dim']}  (signature size)")
print(f"  Layers               : {CONFIG['n_flow_layers']}")
print(f"  Hidden dim           : {CONFIG['hidden_dim']}")
print("Phase 2-A complete ✓")

---
## Phase 3 — The Custom Judges (Loss Functions)

$$\mathcal{L}_{\text{Total}} = \mathcal{L}_{\text{MLE}} + \lambda_1\mathcal{L}_{\text{Dist}} + \lambda_2\mathcal{L}_{\text{ACF}} + \lambda_3\mathcal{L}_{\text{Div}}$$

| Judge | Formula | Purpose |
|---|---|---|
| MLE | $-\mathbb{E}[\log p_Y(y\|S(X))]$ | Core likelihood engine |
| Distribution | $(\mu_{r}-\mu_{f})^2 + (\sigma_{r}-\sigma_{f})^2$ | Match mean & std |
| ACF | $(\rho(Y_{r})-\rho(Y_{f}))^2$ | Match volatility clustering |
| Diversity | $1/(\text{Var}(Y_{\text{fake}})+\epsilon)$ | Prevent mode collapse |

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 3-A  Individual Loss Judges
# ─────────────────────────────────────────────────────────────────

def loss_mle(flow, context, y_real):
    """
    Phase 3 – Judge 1: Maximum Likelihood Estimation (the base engine).

    Minimise the Negative Log-Likelihood:
        L_MLE = -E[ log p_Y(y | S(X)) ]

    The `flow.log_prob` call internally computes:
        log p_Z(f⁻¹(y | S(X))) + log|det(∂f⁻¹/∂y)|
    i.e. the change-of-variables formula.
    """
    log_prob = flow.log_prob(inputs=y_real, context=context)  # (batch,)
    return -log_prob.mean()


def loss_distribution(y_real, y_fake):
    """
    Phase 3 – Judge 2: Distribution matching (mean + std).

        L_Dist = (μ_real - μ_fake)² + (σ_real - σ_fake)²
    """
    mu_r,  sigma_r  = y_real.mean(), y_real.std()
    mu_f,  sigma_f  = y_fake.mean(), y_fake.std()
    return (mu_r - mu_f) ** 2 + (sigma_r - sigma_f) ** 2


def loss_acf(y_real, y_fake):
    """
    Phase 3 – Judge 3: Lag-1 autocorrelation matching (volatility clustering).

        L_ACF = (ρ(Y_real) - ρ(Y_fake))²
        ρ(Y) = Cov(Y_t, Y_{t-1}) / Var(Y)
    """
    def autocorr_lag1(x):
        x = x.squeeze()
        x_mean = x.mean()
        x_c    = x - x_mean
        # Cov(Y_t, Y_{t-1})
        cov    = (x_c[1:] * x_c[:-1]).mean()
        var    = (x_c ** 2).mean() + 1e-8
        return cov / var

    rho_r = autocorr_lag1(y_real)
    rho_f = autocorr_lag1(y_fake)
    return (rho_r - rho_f) ** 2


def loss_diversity(y_fake, eps=1e-6):
    """
    Phase 3 – Judge 4: Anti-mode-collapse / Diversity penalty.

    If all generated samples are identical, Var(Y_fake) → 0 and this
    loss explodes, forcing the model to spread out its predictions.

        L_Div = 1 / (Var(Y_fake) + ε)
    """
    return 1.0 / (y_fake.var() + eps)


def total_loss(flow, context, y_real, cfg):
    """
    Phase 3 – Combined loss:
        L_Total = L_MLE + λ₁·L_Dist + λ₂·L_ACF + λ₃·L_Div
    """
    # Generate fake samples by pushing noise through the conditioned flow
    with torch.no_grad():
        y_fake = flow.sample(num_samples=y_real.shape[0], context=context)

    l_mle  = loss_mle(flow, context, y_real)
    l_dist = loss_distribution(y_real, y_fake)
    l_acf  = loss_acf(y_real, y_fake)
    l_div  = loss_diversity(y_fake)

    total = (
        l_mle
        + cfg['lambda_dist'] * l_dist
        + cfg['lambda_acf']  * l_acf
        + cfg['lambda_div']  * l_div
    )
    return total, l_mle.item(), l_dist.item(), l_acf.item(), l_div.item()


print("Loss functions defined ✓")
print(f"  λ₁ (distribution): {CONFIG['lambda_dist']}")
print(f"  λ₂ (ACF)         : {CONFIG['lambda_acf']}")
print(f"  λ₃ (diversity)   : {CONFIG['lambda_div']}")
print("Phase 3 complete ✓")

---
## Phase 4 — The Training Loop

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 4-A  Optimiser
# ─────────────────────────────────────────────────────────────────
optimizer = optim.Adam(flow.parameters(), lr=CONFIG['learning_rate'])
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CONFIG['epochs'], eta_min=1e-5
)

print(f"Optimiser : Adam  (lr={CONFIG['learning_rate']})")
print(f"Scheduler : CosineAnnealing  (T_max={CONFIG['epochs']})")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 4-B  Training Loop
# ─────────────────────────────────────────────────────────────────
history = {'train_total': [], 'train_mle': [], 'train_dist': [],
           'train_acf':   [], 'train_div': [], 'val_nll':    []}

best_val_nll = float('inf')
best_state   = None

print(f"Starting training for {CONFIG['epochs']} epochs...\n")
print(f"{'Epoch':>6}  {'Train L':>10}  {'MLE':>10}  {'Dist':>8}  {'ACF':>8}  {'Div':>8}  {'Val NLL':>10}")
print("-" * 75)

for epoch in range(1, CONFIG['epochs'] + 1):

    # ── Training ─────────────────────────────────────────────────
    flow.train()
    epoch_totals = []
    epoch_mle    = []
    epoch_dist   = []
    epoch_acf    = []
    epoch_div    = []

    for sig_batch, tgt_batch in train_loader:
        optimizer.zero_grad()

        loss, l_mle, l_dist, l_acf, l_div = total_loss(
            flow, sig_batch, tgt_batch, CONFIG
        )
        loss.backward()

        # Gradient clipping for stability
        nn.utils.clip_grad_norm_(flow.parameters(), max_norm=5.0)
        optimizer.step()

        epoch_totals.append(loss.item())
        epoch_mle.append(l_mle)
        epoch_dist.append(l_dist)
        epoch_acf.append(l_acf)
        epoch_div.append(l_div)

    scheduler.step()

    # ── Validation ───────────────────────────────────────────────
    flow.eval()
    val_nlls = []
    with torch.no_grad():
        for sig_batch, tgt_batch in val_loader:
            nll = loss_mle(flow, sig_batch, tgt_batch)
            val_nlls.append(nll.item())

    train_total = np.mean(epoch_totals)
    train_mle   = np.mean(epoch_mle)
    train_dist  = np.mean(epoch_dist)
    train_acf   = np.mean(epoch_acf)
    train_div   = np.mean(epoch_div)
    val_nll     = np.mean(val_nlls)

    history['train_total'].append(train_total)
    history['train_mle'].append(train_mle)
    history['train_dist'].append(train_dist)
    history['train_acf'].append(train_acf)
    history['train_div'].append(train_div)
    history['val_nll'].append(val_nll)

    # Save best model
    if val_nll < best_val_nll:
        best_val_nll = val_nll
        best_state   = {k: v.cpu().clone() for k, v in flow.state_dict().items()}

    if epoch % CONFIG['log_every'] == 0 or epoch == 1:
        print(f"{epoch:>6}  {train_total:>10.4f}  {train_mle:>10.4f}  "
              f"{train_dist:>8.6f}  {train_acf:>8.6f}  {train_div:>8.4f}  {val_nll:>10.4f}")

# Restore best weights
flow.load_state_dict(best_state)
print(f"\nTraining complete ✓  Best val NLL: {best_val_nll:.4f}")
print("Phase 4 complete ✓")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 4-C  Training Diagnostics Plot
# ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.suptitle('SigFlow-Sim — Training Diagnostics', fontsize=14, fontweight='bold')

plots = [
    ('Total Train Loss',       history['train_total'], 'tab:blue'),
    ('MLE Loss',               history['train_mle'],   'tab:orange'),
    ('Distribution Loss',      history['train_dist'],  'tab:green'),
    ('ACF Loss',               history['train_acf'],   'tab:red'),
    ('Diversity Loss',         history['train_div'],   'tab:purple'),
    ('Validation NLL',         history['val_nll'],     'tab:cyan'),
]

for ax, (title, data, color) in zip(axes.flat, plots):
    ax.plot(data, color=color, lw=1.5)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Epoch')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_diagnostics.png', dpi=120, bbox_inches='tight')
plt.show()
print("Training diagnostics saved ✓")

---
## Phase 5 — Evaluation & Scenario Simulation

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 5-A  Statistical Evaluation on Validation Set
# ─────────────────────────────────────────────────────────────────
flow.eval()

# Collect one generated sample per validation point
all_fake = []
all_real = []

with torch.no_grad():
    for sig_batch, tgt_batch in val_loader:
        y_fake = flow.sample(num_samples=sig_batch.shape[0], context=sig_batch)
        all_fake.append(y_fake.cpu().numpy())
        all_real.append(tgt_batch.cpu().numpy())

y_fake_all = np.concatenate(all_fake).squeeze()
y_real_all = np.concatenate(all_real).squeeze()

print("=" * 55)
print("Validation Set Statistics")
print("=" * 55)
print(f"{'Metric':<20} {'Real':>10} {'Generated':>12}")
print("-" * 45)
print(f"{'Mean':<20} {y_real_all.mean():>10.6f} {y_fake_all.mean():>12.6f}")
print(f"{'Std Dev':<20} {y_real_all.std():>10.6f} {y_fake_all.std():>12.6f}")
print(f"{'Skewness':<20} {pd.Series(y_real_all).skew():>10.4f} {pd.Series(y_fake_all).skew():>12.4f}")
print(f"{'Kurtosis':<20} {pd.Series(y_real_all).kurtosis():>10.4f} {pd.Series(y_fake_all).kurtosis():>12.4f}")
print(f"{'Min':<20} {y_real_all.min():>10.6f} {y_fake_all.min():>12.6f}")
print(f"{'Max':<20} {y_real_all.max():>10.6f} {y_fake_all.max():>12.6f}")

# KS test: does the generated distribution match the real one?
ks_stat, ks_p = kstest(y_fake_all, y_real_all)
print(f"\nKolmogorov-Smirnov test")
print(f"  KS statistic : {ks_stat:.4f}")
print(f"  p-value      : {ks_p:.4f}")
print(f"  Result       : {'PASS (distributions match)' if ks_p > 0.05 else 'FAIL (distributions differ)'}")
print("=" * 55)

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 5-B  Distribution Comparison Plots
# ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('SigFlow-Sim — Distribution Evaluation', fontsize=13, fontweight='bold')

# ── Histogram overlay ────────────────────────────────────────────
axes[0].hist(y_real_all, bins=60, alpha=0.5, density=True,
             color='steelblue', label='Real returns')
axes[0].hist(y_fake_all, bins=60, alpha=0.5, density=True,
             color='tomato', label='Generated returns')
xs = np.linspace(y_real_all.min(), y_real_all.max(), 200)
axes[0].plot(xs, norm.pdf(xs, y_real_all.mean(), y_real_all.std()),
             'k--', lw=1.5, label='Fitted Normal')
axes[0].set_title('Return Distributions')
axes[0].set_xlabel('Log-Return')
axes[0].set_ylabel('Density')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# ── QQ plot ─────────────────────────────────────────────────────
real_q = np.quantile(y_real_all, np.linspace(0.01, 0.99, 100))
fake_q = np.quantile(y_fake_all, np.linspace(0.01, 0.99, 100))
axes[1].scatter(real_q, fake_q, s=15, alpha=0.7, color='mediumorchid')
lim_min = min(real_q.min(), fake_q.min())
lim_max = max(real_q.max(), fake_q.max())
axes[1].plot([lim_min, lim_max], [lim_min, lim_max], 'k--', lw=1.5)
axes[1].set_title('Q-Q Plot (Real vs Generated)')
axes[1].set_xlabel('Real Quantiles')
axes[1].set_ylabel('Generated Quantiles')
axes[1].grid(True, alpha=0.3)

# ── Autocorrelation comparison ────────────────────────────────────
max_lag = 20
acf_real = [pd.Series(y_real_all).autocorr(lag=k) for k in range(1, max_lag+1)]
acf_fake = [pd.Series(y_fake_all).autocorr(lag=k) for k in range(1, max_lag+1)]
lags = range(1, max_lag+1)
axes[2].bar([l - 0.2 for l in lags], acf_real, width=0.35,
            alpha=0.7, label='Real ACF', color='steelblue')
axes[2].bar([l + 0.2 for l in lags], acf_fake, width=0.35,
            alpha=0.7, label='Generated ACF', color='tomato')
axes[2].axhline(0, color='black', lw=0.8)
axes[2].set_title('Autocorrelation Structure')
axes[2].set_xlabel('Lag (days)')
axes[2].set_ylabel('Autocorrelation')
axes[2].legend(fontsize=8)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('distribution_evaluation.png', dpi=120, bbox_inches='tight')
plt.show()
print("Distribution evaluation complete ✓")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 5-C  Forward Scenario Simulation
#
# We take the LAST `window_size` days of real data as the seed,
# compute its Signature, and autoregressively generate N scenarios.
# ─────────────────────────────────────────────────────────────────
def simulate_forward(
    flow,
    seed_returns : np.ndarray,   # shape (window_size,)
    scaler,
    n_scenarios  : int,
    n_steps      : int,
    sig_depth    : int,
    sig_size     : int,
    device,
) -> np.ndarray:
    """
    Autoregressively generate `n_scenarios` paths of `n_steps` returns.

    At each step:
      1. Compute Signature of the current window.
      2. Sample a return from the conditioned flow.
      3. Append the generated return to the window (roll).

    Returns an array of shape (n_scenarios, n_steps).
    """
    flow.eval()
    all_scenarios = np.zeros((n_scenarios, n_steps), dtype=np.float32)

    for s in range(n_scenarios):
        window = seed_returns.copy()  # (W,)

        for step in range(n_steps):
            # Step 1 – Compute Signature
            path    = build_path_with_time(window)
            sig_vec = compute_signature(path, sig_depth)  # (sig_size,)

            # Step 2 – Scale (use the fitted training scaler)
            sig_scaled = scaler.transform(sig_vec.reshape(1, -1)).astype(np.float32)
            sig_t      = torch.tensor(sig_scaled).to(device)  # (1, sig_size)

            # Step 3 – Sample from the flow: Z → Y
            with torch.no_grad():
                y_gen = flow.sample(num_samples=1, context=sig_t)  # (1, 1)
            r = y_gen.cpu().numpy().squeeze()

            all_scenarios[s, step] = r

            # Step 4 – Roll the window forward
            window = np.roll(window, -1)
            window[-1] = r

    return all_scenarios


# Use the last window of real returns as seed
seed = log_returns[-CONFIG['window_size']:]
print(f"Seed window (last {CONFIG['window_size']} trading days):")
print(f"  Mean return : {seed.mean():.5f}")
print(f"  Std return  : {seed.std():.5f}")
print(f"\nGenerating {CONFIG['n_scenarios']} scenarios × {CONFIG['n_steps']} steps...")

scenarios = simulate_forward(
    flow         = flow,
    seed_returns = seed,
    scaler       = scaler,
    n_scenarios  = CONFIG['n_scenarios'],
    n_steps      = CONFIG['n_steps'],
    sig_depth    = CONFIG['sig_depth'],
    sig_size     = CONFIG['context_dim'],
    device       = DEVICE,
)

print(f"Scenarios shape: {scenarios.shape}  (n_scenarios × n_steps)")
print("Simulation complete ✓")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 5-D  Convert return scenarios → price paths & visualise
# ─────────────────────────────────────────────────────────────────
last_price = float(prices.iloc[-1])

# Convert log-returns to price paths
# P_{t+k} = P_t · exp(sum of log-returns from t to t+k)
cumulative_returns = np.cumsum(scenarios, axis=1)        # (n_scenarios, n_steps)
price_paths = last_price * np.exp(cumulative_returns)    # (n_scenarios, n_steps)

# Include day 0 (current price) by prepending
day0 = np.full((CONFIG['n_scenarios'], 1), last_price)
price_paths = np.hstack([day0, price_paths])             # (n_scenarios, n_steps+1)

# ── Percentile bands ────────────────────────────────────────────
p10  = np.percentile(price_paths, 10,  axis=0)
p25  = np.percentile(price_paths, 25,  axis=0)
p50  = np.percentile(price_paths, 50,  axis=0)
p75  = np.percentile(price_paths, 75,  axis=0)
p90  = np.percentile(price_paths, 90,  axis=0)

steps = np.arange(CONFIG['n_steps'] + 1)

# ── Plot ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(
    f'SigFlow-Sim — {CONFIG["n_scenarios"]} Forward Scenarios '
    f'({CONFIG["n_steps"]} trading days)',
    fontsize=13, fontweight='bold'
)

# Left: individual path spaghetti
for i in range(min(CONFIG['n_scenarios'], 30)):
    axes[0].plot(steps, price_paths[i], lw=0.6, alpha=0.4, color='steelblue')
axes[0].plot(steps, p50, 'k-', lw=2.0, label='Median')
axes[0].axhline(last_price, color='red', lw=1.5, ls='--', label=f'Current price ({last_price:.2f})')
axes[0].set_title('Individual Scenarios')
axes[0].set_xlabel('Trading Days Forward')
axes[0].set_ylabel('Price (USD)')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Right: fan / percentile bands
axes[1].fill_between(steps, p10, p90, alpha=0.2, color='steelblue', label='10–90th pct')
axes[1].fill_between(steps, p25, p75, alpha=0.35, color='steelblue', label='25–75th pct')
axes[1].plot(steps, p50, 'k-',   lw=2.0, label='Median')
axes[1].plot(steps, p10, 'b--',  lw=1.0)
axes[1].plot(steps, p90, 'b--',  lw=1.0)
axes[1].axhline(last_price, color='red', lw=1.5, ls='--', label=f'Current ({last_price:.2f})')
axes[1].set_title('Percentile Fan Chart')
axes[1].set_xlabel('Trading Days Forward')
axes[1].set_ylabel('Price (USD)')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('forward_scenarios.png', dpi=120, bbox_inches='tight')
plt.show()
print("Forward scenario plot saved ✓")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 5-E  Risk Metrics from the Simulation
# ─────────────────────────────────────────────────────────────────
# Final price at end of simulation
final_prices  = price_paths[:, -1]
final_returns = (final_prices / last_price - 1) * 100   # percent

var_95 = np.percentile(final_returns, 5)   # Value-at-Risk (95 % confidence)
var_99 = np.percentile(final_returns, 1)   # VaR 99%
cvar   = final_returns[final_returns <= var_95].mean()  # Expected Shortfall

print("=" * 55)
print(f"Risk Metrics — {CONFIG['n_steps']}-day Horizon")
print("=" * 55)
print(f"  Current price        : ${last_price:.2f}")
print(f"  Median final price   : ${np.median(final_prices):.2f}  "
      f"({np.median(final_returns):+.2f}%)")
print(f"  Mean final price     : ${final_prices.mean():.2f}  "
      f"({final_returns.mean():+.2f}%)")
print(f"  10th pct price       : ${np.percentile(final_prices,10):.2f}")
print(f"  90th pct price       : ${np.percentile(final_prices,90):.2f}")
print(f"  VaR 95%  (5th pct)   : {var_95:+.2f}%")
print(f"  VaR 99%  (1st pct)   : {var_99:+.2f}%")
print(f"  CVaR / ES (95%)      : {cvar:+.2f}%")
print(f"  Prob of gain         : {(final_returns > 0).mean()*100:.1f}%")
print(f"  Prob of >10% gain    : {(final_returns > 10).mean()*100:.1f}%")
print(f"  Prob of >10% loss    : {(final_returns < -10).mean()*100:.1f}%")
print("=" * 55)

# Final return distribution plot
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(final_returns, bins=40, density=True,
        color='steelblue', alpha=0.7, edgecolor='white', lw=0.5)
ax.axvline(var_95, color='orange', lw=2, ls='--', label=f'VaR 95% ({var_95:.1f}%)')
ax.axvline(var_99, color='red',    lw=2, ls='--', label=f'VaR 99% ({var_99:.1f}%)')
ax.axvline(np.median(final_returns), color='black', lw=2,
           label=f'Median ({np.median(final_returns):.1f}%)')
ax.set_title(f'Distribution of {CONFIG["n_steps"]}-day Returns ({CONFIG["n_scenarios"]} scenarios)',
             fontsize=12)
ax.set_xlabel('Return (%)')
ax.set_ylabel('Density')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('risk_metrics.png', dpi=120, bbox_inches='tight')
plt.show()
print("Risk metrics plot saved ✓")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 6  Save model weights
# ─────────────────────────────────────────────────────────────────
torch.save({
    'model_state_dict': flow.state_dict(),
    'config'          : CONFIG,
    'scaler_mean'     : scaler.mean_,
    'scaler_scale'    : scaler.scale_,
    'best_val_nll'    : best_val_nll,
}, 'sigflow_sim.pt')

print("Model saved to sigflow_sim.pt ✓")
print("\n" + "=" * 55)
print(" SigFlow-Sim complete!")
print("=" * 55)
print("  Phase 1 — Path Signatures       ✓")
print("  Phase 2 — Normalizing Flow       ✓")
print("  Phase 3 — Custom Loss Judges     ✓")
print("  Phase 4 — Training Loop          ✓")
print("  Phase 5 — Evaluation & Scenarios ✓")